This notebook focuses on building a customer churn prediction model using XGBoost, combined with structured and reusable feature engineering techniques implemented through custom Scikit-learn transformers.

## Importing data:

In [45]:
import pandas as pd

#Training data
train_path = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
train_df = pd.read_csv(train_path)

#Test data
test_path = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
test_df = pd.read_csv(test_path)

In [46]:
#looking for null values
print('null values in training data')
print(train_df.isna().sum())

print('\n')

print('null values in test data')
print(test_df.isna().sum())

null values in training data
id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


null values in test data
id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64


In [47]:
target = 'Churn'

#encoding the target 
train_df[target] = train_df[target].map({"No": 0, "Yes": 1}) 

#dropping redundant features ('id')
train_df = train_df.drop('id', axis=1)
X_test = test_df.drop('id', axis=1)

#seperating features and label
X = train_df.drop(target, axis=1)
y = train_df[target]

Encoding all the categorical columns using LabelEncoder, and later we'll perform feature engineering on them.

In [48]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include='object').columns.to_list()
encoder = {} #just in case we have to reuse these encoders

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    X_test[col] = le.transform(X_test[col])
    encoder[col] = le

Now all our categorical/object features are converted to numeric, we can perform feature engineering without any worry.

## Feature Engineering:
Trying the best feature engineering methods to create new and more informative features. This section will consist:

* Binning 'tenure'
* Groupby categorical features and aggrigate statistics of numerical features
* Frequency Encoding on cat_cols

We'll create custom transformers using BaseEstimator and TransformerMixin, what ScikitLearn does is:

Call fit() on your transformer Then immediately call transform() on that SAME training data, without an explicit call

The transformers i'm using here were created in the Heart Disease prediction competition, so we'll reuse the same code, though i'll still explin what they do...

### Binning transformer:

In [49]:
from sklearn.base import BaseEstimator, TransformerMixin

class Binning(BaseEstimator, TransformerMixin):
    def __init__(self, col_to_bin, num_bins, new_col_name ,labels=None):
        self.col_to_bin = col_to_bin
        self.num_bins = num_bins
        self.labels = labels
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        X = X.copy()
        _, self.bin_edges = pd.cut(X[self.col_to_bin], bins=self.num_bins, labels=False, retbins=True) #the last attribute ensures that the same bins are used to bin teh test data during transform
        return self

    def transform(self,X):
        X = X.copy() 
        X[self.new_col_name] = pd.cut(X[self.col_to_bin], bins=self.bin_edges, labels=False)
        return X

### GroupMean Encoder:
Applying group-mean encoding on Monthly and Total Charges columns, by grouping them using 'tenure bins' created using binning transformer

In [50]:
class GroupMeanEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, groupby_col, agg_col, new_col_name):
        self.groupby_col = groupby_col
        self.agg_col = agg_col
        self.new_col_name = new_col_name

    def fit(self,X,y=None):
        self.means = X.groupby(self.groupby_col,observed=True)[self.agg_col].mean()
        return self

    def transform(self,X):
        X = X.copy()
        X[self.new_col_name] = X[self.groupby_col].map(self.means)
        return X

### Frequency Encoding:
Frequency Encoding is a way to convert a categorical feature into numbers by replacing each category with how often it appears in the dataset.

In [51]:
class FreqEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, normalize=True):
        self.cat_cols = cat_cols
        self.normalize = normalize
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cat_cols:
            self.freq_maps[col] = X[col].value_counts(normalize=self.normalize)
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.cat_cols:
            X[col + '_freq'] = X[col].map(self.freq_maps[col])
            X[col + '_freq'] = X[col + '_freq'].fillna(0) 
        return X

## Preprocessing pipeline:

In [52]:
from sklearn.pipeline import Pipeline

preprocessor = Pipeline([
    ('Binning', Binning(col_to_bin='tenure', num_bins=3, new_col_name='tenure_bins')),
    ('GroupMeanEncoder_BP', GroupMeanEncoder(groupby_col='tenure_bins', agg_col='MonthlyCharges', new_col_name='X1')),
    ('GroupMeanEncoder_Cholesterol', GroupMeanEncoder(groupby_col='tenure_bins', agg_col='TotalCharges', new_col_name='X2')),
    ('FreqEncoding', FreqEncoder(cat_cols=cat_cols)),
])

The hyperprameters were found using Optuna during privious experiments of mine in the same competition:

In [56]:
best_xgb_params = {'n_estimators': 3488, 
                   'learning_rate': 0.04215348921376158,
                   'max_depth':3, 
                   'min_child_weight': 3, 
                   'gamma': 0.373740919749451, 
                   'subsample': 0.8908198617400997, 
                   'colsample_bytree': 0.7125429511845258,
                   'reg_alpha': 5.2763994068891605e-08, 
                   'reg_lambda': 1.0023283765023319e-07}

## Training on training dataset:

In [57]:
from xgboost import XGBClassifier

best_model = XGBClassifier(
    **best_xgb_params,
    random_state= 42,
    eval_metric= "logloss",
    tree_method= "hist",   
    verbosity= 0,
    device='cuda',
    enable_categorical=True
)

final_pipeline = Pipeline([
        ('prep', preprocessor),
        ('model', best_model)
    ])

final_pipeline.fit(X,y)

Pipeline(steps=[('prep',
                 Pipeline(steps=[('Binning',
                                  Binning(col_to_bin='tenure',
                                          new_col_name='tenure_bins',
                                          num_bins=3)),
                                 ('GroupMeanEncoder_BP',
                                  GroupMeanEncoder(agg_col='MonthlyCharges',
                                                   groupby_col='tenure_bins',
                                                   new_col_name='X1')),
                                 ('GroupMeanEncoder_Cholesterol',
                                  GroupMeanEncoder(agg_col='TotalCharges',
                                                   groupby_col='tenure_bins',
                                                   new_col_name='X2')),
                                 ('...
                               gamma=2.2801752923971543, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.04750177835534781, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=4,
                               max_leaves=None, min_child_weight=4, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=2642, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [58]:
y_pred = final_pipeline.predict_proba(X_test)[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [59]:
submission = pd.DataFrame({
    'id': test_df['id'],
    target: y_pred
})

submission.to_csv('submission.csv', index=False)